# 실습 02 · OCR 방법 1과 방법 2 비교

방법 1은 해설 코드, 방법 2는 직접 작성한 코드의 수정본입니다. 아래 셀을 순서대로 실행하면 실제 문서 3장을 두 방식으로 인식하고 페이지별 결과를 비교합니다.

Python 커널에 pandas, OpenCV, pytesseract, nbformat이 필요하며, 별도로 Tesseract 엔진과 eng·kor 언어팩이 설치되어 있어야 합니다. 노트북은 저장소 내부에서 실행하세요. 입력 목록은 임시 폴더에 만들며 기존 CSV를 덮어쓰지 않습니다.

| 항목 | 방법 1: 목록 일괄 처리 | 방법 2: 이미지별 처리 |
|---|---|---|
| 전달 입력 | 이미지 경로가 담긴 텍스트 파일 | OpenCV로 읽고 이진화한 배열 |
| 엔진 호출 | 전체 목록에 1회 | 이미지마다 1회 |
| 별도 전처리 | 없음 (엔진 내부 처리는 수행) | Gray 변환 + Otsu 이진화 |
| 언어 | eng | kor+eng |
| 엔진 설정 | --psm 6 | --oem 3 --psm 6 |
| 페이지 번호 | 엔진이 순서대로 부여 | 직접 1, 2, 3 지정 |
| 결과 병합 | 한 번에 DataFrame 반환 | pd.concat으로 병합 |
| 텍스트 읽기 | pandas 기본 설정 | 문자열 유지, 기본 NA 변환 해제 |

`--psm 6`은 한 개의 균일한 텍스트 블록을 가정합니다. `--oem 3`은 사용 가능한 엔진에 따른 기본 선택입니다. 방법 2가 항상 더 정확한 것은 아닙니다.


In [1]:
from pathlib import Path
from tempfile import TemporaryDirectory
from time import perf_counter
import os
import cv2
import pandas as pd
import pytesseract
from IPython.display import display

start = Path.cwd().resolve()
project = next((p for p in [start, *start.parents]
                if (p / '20261109/01_basic/02_ocr.py').is_file()), None)
if project is None:
    raise FileNotFoundError('저장소 내부에서 실행하세요.')
base = project / '20261109/01_basic'
paths = [base / f'doc{i:02d}.png' for i in range(1, 4)]
pytesseract.pytesseract.tesseract_cmd = os.environ.get('TESSERACT_CMD', 'tesseract')
print('언어팩:', pytesseract.get_languages(config=''))


언어팩: ['eng', 'kor', 'osd']


## 방법 1 · 경로 목록을 한 번에 전달

Tesseract가 목록에 있는 이미지들을 읽고 `page_num`을 구분합니다. Python에서 이미지를 따로 읽을 필요가 없습니다.


In [2]:
begin = perf_counter()
with TemporaryDirectory() as temp:
    input_list = Path(temp) / 'inputs.txt'
    input_list.write_text('\n'.join(map(str, paths)), encoding='utf-8')
    method1 = pytesseract.image_to_data(
        str(input_list), lang='eng', config='--psm 6',
        output_type=pytesseract.Output.DATAFRAME)
seconds1 = perf_counter() - begin


## 방법 2 · 전처리 후 이미지별 OCR

원래 코드에서 각 이미지의 `page_num`은 모두 1이었습니다. `concat()`은 이 번호를 바꾸지 않으므로 그대로 합치면 마지막 `groupby('page_num')`에서 세 문서가 하나로 묶입니다. 아래에서는 이미지 순서로 페이지 번호를 지정합니다.

`ignore_index=True`는 행 인덱스만 새로 매깁니다. 페이지 번호를 수정하는 기능은 없습니다. 읽기 실패 검사와 빈 입력 처리도 추가했습니다.


In [3]:
begin = perf_counter()
ocr_results = []
for page_number, path in enumerate(paths, start=1):
    image = cv2.imread(str(path))
    if image is None:
        raise FileNotFoundError(f'이미지를 읽을 수 없습니다: {path}')
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY | cv2.THRESH_OTSU)
    ocr = pytesseract.image_to_data(
        binary, lang='kor+eng', config='--oem 3 --psm 6',
        output_type=pytesseract.Output.DATAFRAME,
        pandas_config={'keep_default_na': False, 'dtype': {'text': str}})
    ocr['page_num'] = page_number
    ocr_results.append(ocr)
method2 = pd.concat(ocr_results, ignore_index=True) if ocr_results else pd.DataFrame(
    columns=['page_num', 'text', 'conf'])
seconds2 = perf_counter() - begin


## 같은 기준으로 정리하고 결과 비교

방법 2는 빈 문자열을 유지하므로 `dropna()`만으로 빈 텍스트를 지울 수 없습니다. 두 방식 모두 공백 문자열과 `conf < 0`인 구조 행을 제거합니다. 기본 NA 변환을 끄면 `NA` 같은 문자열이 결측값으로 처리되는 것을 방지할 수 있습니다.


In [4]:
def clean(words):
    words = words.dropna(subset=['text'])
    return words[words['text'].astype(str).str.strip().ne('') & (words['conf'] >= 0)].copy()

records = []
for label, words in [('방법 1', method1), ('방법 2', method2)]:
    words = clean(words)
    for page, rows in words.groupby('page_num', sort=True):
        records.append({'방법': label, '페이지': page, '단어 수': len(rows),
                        '평균 confidence': round(rows['conf'].mean(), 2),
                        '텍스트': ' '.join(rows['text'].astype(str))})
display(pd.DataFrame(records))
display(pd.DataFrame({'방법': ['방법 1', '방법 2'],
                      '엔진 호출 수': [1, len(paths)],
                      '전체 처리 시간(초)': [seconds1, seconds2]}))
assert set(method1['page_num']) == set(method2['page_num']) == {1, 2, 3}
print('두 방식 모두 페이지 번호 1, 2, 3을 구분합니다.')


,방법,페이지,단어 수,평균 confidence,텍스트
0,방법 1,1,27,95.44,INVOICE DOCO1 DATE: 2026-11-01 SUPPLIER: BLUE ...
1,방법 1,2,27,95.13,INVOICE DOCO2 DATE: 2026-11-02 SUPPLIER: PINE ...
2,방법 1,3,27,94.55,INVOICE DOCO3 DATE: 2026-11-03 SUPPLIER: ALPHA...
3,방법 2,1,27,95.15,INVOICE 00601 DATE: 2026-11-01 SUPPLIER: BLUE ...
4,방법 2,2,27,94.99,INVOICE 00662 DATE: 2026-11-02 SUPPLIER: PINE ...
5,방법 2,3,27,94.87,INVOICE 00603 DATE: 2026-11-03 SUPPLIER: ALPHA...


,방법,엔진 호출 수,전체 처리 시간(초)
0,방법 1,1,0.266781
1,방법 2,3,0.467228


두 방식 모두 페이지 번호 1, 2, 3을 구분합니다.


## 결과를 해석할 때

- 방법 1은 여러 페이지를 한 번 호출로 처리한다는 실습 목적에 맞습니다. 엔진 실행 횟수를 줄일 수 있지만 실제 속도는 환경과 문서에 따라 확인해야 합니다.
- 방법 2는 이미지마다 전처리나 인식 설정을 바꾸기 쉽습니다. Otsu 이진화가 배경 잡음을 줄이기도 하지만 얇은 획을 지울 수도 있습니다.
- 위 비교는 전처리, 언어, 텍스트 읽기 설정이 동시에 다릅니다. 결과 차이를 전처리 하나의 효과라고 판단할 수 없습니다. 전처리 효과만 비교하려면 언어와 OCR·pandas 설정을 동일하게 맞추고 원본/이진화 입력만 바꾸세요.
- confidence는 정답 확률이 아닙니다. 평균 confidence나 단어 수가 높다고 정확도가 높다고 결론 내릴 수 없습니다. 정확도 평가는 정답 텍스트와의 비교가 필요합니다.
- 시간은 전처리와 결과 변환 등을 포함한 1회 측정입니다. 성능을 평가하려면 같은 조건에서 반복 측정하세요.
